# Stable Diffusion test UI (SD 1.5 / SDXL) on Colab

Load **one** Stable Diffusion 1.5 or SDXL checkpoint (`.safetensors`) from Hugging Face or Civitai, keep it on the GPU, and generate images from a small web UI that opens right here in the notebook.

**How to use**
1. **Runtime → Change runtime type → GPU (L4 recommended)**. A T4 works but is slower and tighter on memory for SDXL.
2. Fill in **1 · Settings** (model family + model link), then run cells **1 → 5** in order (or *Runtime → Run all*).
3. The UI appears under cell **5**. It is usable immediately — you can queue prompts while the model is still loading; its status pill shows loading progress and any errors.

The notebook only sets things up and starts the app; it stays quiet and never shows your prompts or images. (Troubleshooting: the server log is in `/content/sdmodeltester/server.log`.)

**Model links that work**
- Hugging Face file: `https://huggingface.co/<org>/<repo>/blob/main/<file>.safetensors`
- Civitai model page: `https://civitai.com/models/<id>?modelVersionId=<versionId>` (or the page without `modelVersionId` for the newest version), or a download link `https://civitai.com/api/download/models/<versionId>`
- A file already in Colab or on your Google Drive, e.g. `/content/drive/MyDrive/models/model.safetensors`

You must choose the **model family** yourself (`sd15` or `sdxl`; Pony/Illustrious/NoobAI are `sdxl`).

**Tokens (optional)** — some models need a Hugging Face or Civitai token. Add them as **Colab Secrets** (🔑 icon in the left sidebar) named `HF_TOKEN` / `CIVITAI_TOKEN` and allow notebook access. Secrets are never saved in the notebook, so it stays safe to share. Typing a token into the settings form also works, but then **don't share your copy**.

**Prompt syntax**: `{white | black}` picks one option per image; `(white hair:1.2)` weights a phrase. Seed `-1` = random per image.

**Privacy**: images stay on this Colab machine (no prompt metadata is written into the PNGs). Use cell 6 to download them, and the UI's *Danger zone* to delete them. Everything is gone when the runtime is deleted.

*Note:* Colab's resource and usage policies apply; free-tier sessions may be limited or disconnected. Keep this tab open while you use the UI.

In [ ]:
# @title 1 · Settings { display-mode: "form" }
MODEL_FAMILY = "sdxl"  # @param ["sd15", "sdxl"]
MODEL_SOURCE = ""  # @param {type:"string"}
# @markdown ↑ Hugging Face / Civitai link, or a path to a `.safetensors` file (e.g. on Drive).
SAVE_MODELS_TO_DRIVE = False  # @param {type:"boolean"}
# @markdown ↑ Keep downloaded models in Google Drive (`MyDrive/sdmodeltester/models`) so the next session skips the download.
HF_TOKEN = ""  # @param {type:"string"}
CIVITAI_TOKEN = ""  # @param {type:"string"}
# @markdown ↑ Optional. Prefer Colab Secrets named `HF_TOKEN` / `CIVITAI_TOKEN` (🔑 sidebar) — they are used first.
# @markdown ---
# @markdown **Advanced**
OPTIMIZATION = "baseline"  # @param ["baseline", "compile", "compile-max"]
PORT = 8000  # @param {type:"integer"}
DEMO_MODE = False  # @param {type:"boolean"}
# @markdown ↑ Try the UI without a GPU or model (placeholder images).
REPO_URL = "https://github.com/ashwin2rai/sdmodeltester"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
WORK_DIR = "/content"

if MODEL_FAMILY not in ("sd15", "sdxl"):
    raise ValueError("MODEL_FAMILY must be 'sd15' or 'sdxl'")
if not DEMO_MODE and not MODEL_SOURCE.strip():
    print("⚠️  MODEL_SOURCE is empty: paste a model link above before running cell 4.")
print(f"Settings: {MODEL_FAMILY} · optimization {OPTIMIZATION} · port {PORT}"
      + (" · DEMO MODE" if DEMO_MODE else ""))

In [ ]:
# @title 2 · Get the code and check the GPU { display-mode: "form" }
import importlib
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(WORK_DIR) / "sdmodeltester"


def _git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{result.stderr}")
    return result.stdout.strip()


if not (REPO_DIR / ".git").exists():
    _git("clone", "-q", REPO_URL, str(REPO_DIR))
_git("-C", str(REPO_DIR), "fetch", "-q", "--depth", "1", "origin", REPO_REF)
_git("-C", str(REPO_DIR), "checkout", "-q", "--force", "FETCH_HEAD")
COMMIT = _git("-C", str(REPO_DIR), "rev-parse", "--short", "HEAD")

if str(REPO_DIR / "notebooks") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "notebooks"))
import colab_utils as cu  # noqa: E402

cu = importlib.reload(cu)
RUNTIME = cu.runtime_preflight()
print(f"✓ Code {COMMIT} · {RUNTIME['summary']}")
for warning in RUNTIME["warnings"]:
    print(f"⚠️  {warning}")
if not DEMO_MODE and not RUNTIME["cuda"]:
    raise RuntimeError(
        "No GPU: Runtime → Change runtime type → choose a GPU (L4), then run all cells again."
    )

In [ ]:
# @title 3 · Install and check { display-mode: "form" }
# Uses Colab's own Python, CUDA and PyTorch; only app-level libraries are installed.
TORCH = cu.install_requirements(REPO_DIR)
doctor = [sys.executable, "-m", "src.cli", "doctor", "--model-family", MODEL_FAMILY]
if DEMO_MODE:
    doctor.append("--mock")
cu.run(doctor, "Environment check", cwd=REPO_DIR)  # prints details only on failure
print(f"✓ Installed and checked (torch {TORCH} untouched)")

In [ ]:
# @title 4 · Download the model { display-mode: "form" }
if DEMO_MODE:
    MODEL_PATH = None
    print("✓ Demo mode: no model needed")
else:
    if SAVE_MODELS_TO_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        MODELS_DIR = Path("/content/drive/MyDrive/sdmodeltester/models")
    else:
        MODELS_DIR = REPO_DIR / "models"
    status = cu.LiveLine()  # one updating line; warnings stay visible
    MODEL_PATH = cu.fetch_checkpoint(
        MODEL_SOURCE,
        MODEL_FAMILY,
        MODELS_DIR,
        hf_token=cu.get_secret("HF_TOKEN", HF_TOKEN),
        civitai_token=cu.get_secret("CIVITAI_TOKEN", CIVITAI_TOKEN),
        log=status,
    )
    status(f"✓ Model: {MODEL_PATH}")

In [ ]:
# @title 5 · Start the server and open the UI { display-mode: "form" }
# The UI opens immediately; the model loads, optimizes and warms up in the background
# (the UI's status pill shows progress and errors). The notebook stays quiet.
LOG_FILE = REPO_DIR / "server.log"
PID_FILE = REPO_DIR / "server.pid"
serve = [sys.executable, "-m", "src.cli", "serve", "--model-family", MODEL_FAMILY,
         "--port", str(PORT), "--optimization", OPTIMIZATION]
serve += ["--mock"] if DEMO_MODE else ["--model", str(MODEL_PATH)]
cu.start_server(serve, cwd=REPO_DIR, log_file=LOG_FILE, pid_file=PID_FILE, port=PORT,
                log=lambda message: None)
cu.show_ui(PORT)
print("✓ UI running")

## Optional

In [ ]:
# @title 6 · Download your images (optional) { display-mode: "form" }
cu.zip_outputs(REPO_DIR / "outputs", Path(WORK_DIR) / "sdmodeltester-outputs")

In [ ]:
# @title 7 · Benchmark (optional, for tuning — stops the UI) { display-mode: "form" }
# Compares optimization profiles and runs functional checks (objectives/SPEC.md §23).
# Afterwards, run cell 5 again to restart the UI.
PROFILES = "baseline,compile"  # @param {type:"string"}
FUNCTIONAL_CHECKS = True  # @param {type:"boolean"}
cu.stop_server(PID_FILE, PORT, log=lambda message: None)
print("Benchmarking with the UI stopped — this can take 10–30 minutes…")
BENCHMARK = cu.run_benchmark(REPO_DIR, MODEL_FAMILY, model_path=MODEL_PATH, profiles=PROFILES,
                             functional=FUNCTIONAL_CHECKS, mock=DEMO_MODE)
cu.show_benchmark(BENCHMARK)
print(f"Report: {BENCHMARK['report']} (+ .json). Run cell 5 to restart the UI.")

In [ ]:
# @title 8 · Stop the server { display-mode: "form" }
stopped = cu.stop_server(PID_FILE, PORT, log=lambda message: None)
print("✓ UI stopped" if stopped else "The UI was not running.")